# 11 - DeBERTa-v3 NLI Semantic Validation

**Objective**: Quantify the logical entailment between retrieved literature passages (premises) and generated scientific propositions (hypotheses) using a cross-encoder Natural Language Inference (NLI) model (`cross-encoder/nli-deberta-v3-small`).

**Inputs**:
- `outputs/generated_hypotheses.json`
- `outputs/retrieved_gap_evidence.json`

**Outputs**:
- `outputs/nli_results.json` (Entailment, Neutral, Contradiction probability distributions)

**Production Code Reused**:
- `src/phase2/nli_validation.py`


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.phase2.nli_validation import NLIValidator, load_nli_results

print("NLI validation environment initialized.")


## 1. Load Verified Cross-Encoder NLI Findings
We examine the distribution of premise-hypothesis probability triples:
- **Entailment**: Supporting literature directly implies or entails the proposition.
- **Neutral**: The proposition is plausible but not strictly entailed by this single passage.
- **Contradiction**: The proposition conflicts with empirical findings in the passage.


In [ ]:
with open("outputs/nli_results.json", "r", encoding="utf-8") as f:
    nli_data = json.load(f)

print(f"Total NLI Evaluated Pairs: {len(nli_data)}")

# Extract entailment scores
entailments = [item.get("entailment", 0.0) for item in nli_data]
neutrals = [item.get("neutral", 0.0) for item in nli_data]
contradictions = [item.get("contradiction", 0.0) for item in nli_data]

print(f"Mean Entailment Probability:    {np.mean(entailments):.4f}")
print(f"Mean Neutral Probability:       {np.mean(neutrals):.4f}")
print(f"Mean Contradiction Probability: {np.mean(contradictions):.4f}")


## 2. NLI Probability Distribution Plot


In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(entailments, bins=20, alpha=0.7, color='#0d9488', label='Entailment', edgecolor='black')
plt.hist(contradictions, bins=20, alpha=0.7, color='#e11d48', label='Contradiction', edgecolor='black')
plt.title("DeBERTa-v3 Cross-Encoder Probability Distribution", fontsize=12, fontweight='bold')
plt.xlabel("Softmax Probability", fontsize=10)
plt.ylabel("Pair Count", fontsize=10)
plt.legend(loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
